In [1]:
import os
import glob
import pandas as pd

In [2]:
# Find all file pathways containing the core association data
association_files = glob.glob("*_variants.csv")

print(f"Found {len(association_files)} files to process.")

# Specific variant consequences that represent protein-coding exons
coding_terms = ['missense_variant', 'synonymous_variant', 'frameshift_variant', 'stop_gained', 'start_lost', 'inframe_insertion', 'inframe_deletion', 'protein_altering_variant']
pattern = '|'.join(coding_terms)

# Exact layout requirements requested by user
columns_mapping = {
    'base_pair_location': 'Location',
    'effect_allele': 'affected allele',
    'other_allele': 'other allele',
    'Consequence': 'consequences',
    'Protein_position': 'protein position',
    'Amino_acids': 'Amino acid change',
    'Codons': 'codons',
    'effect_allele_frequency': 'allele frequency',
    'p_value': 'p value',
    'neg_log_10_p_value': 'negative logP value'
}

for assoc_file in association_files:
    # Deduce prefixes dynamically for either file pattern naming style
    prefix = assoc_file.replace("_variants.csv", "")
    
    # Check for possible annotation file names (handles case variations if any)
    ann_file = f"{prefix}_annotated.csv"
    if not os.path.exists(ann_file):
        ann_file = f"{prefix}_annotated.csv"
        
    if not os.path.exists(ann_file):
        print(f"⚠️ Warning: Missing matching annotation file for {prefix}. Skipping...")
        continue
        
    print(f"Processing: {prefix}...")
    
    try:
        # 1. Load the summary metrics dataset (auto-detects delimiter style)
        sep_char = '\t' if assoc_file.endswith('.tsv') else ','
        with open(assoc_file, 'r') as test_f:
            first_line = test_f.readline()
            if '\t' in first_line: sep_char = '\t'
            
        df_gene = pd.read_csv(assoc_file, sep=sep_char) 
        df_gene['Location_str'] = df_gene['chromosome'].astype(str) + ':' + df_gene['base_pair_location'].astype(str)
        
        # 2. Load VEP annotations dataset (skipping the 36 metadata header rows)
        df_ann = pd.read_csv(ann_file, sep='\t', skiprows=36)
        df_ann.rename(columns={'#Uploaded_variation': 'Uploaded_variation'}, inplace=True)
        
        # 3. Clean placeholder hyphens into clean empty strings
        for col in df_ann.columns:
            if df_ann[col].dtype == object:
                df_ann[col] = df_ann[col].apply(lambda x: '' if str(x).strip() == '-' else x)
                
        # 4. CRITICAL FIXES: Keep ONLY Canonical transcripts & Coding Exon variants
        df_ann_canonical = df_ann[df_ann['CANONICAL'] == 'YES']
        df_coding = df_ann_canonical[df_ann_canonical['Consequence'].str.contains(pattern, case=False, na=False)]
        
        # 5. Merge datasets together precisely
        merged = pd.merge(df_coding, df_gene, left_on='Location', right_on='Location_str', how='inner')
        
        # 6. Apply final structural column selection and export
        if not merged.empty:
            final_df = merged[list(columns_mapping.keys())].rename(columns=columns_mapping)
            output_name = f"{prefix.lower()}_exon_coding_variants.csv"
            final_df.to_csv(output_name, index=False)
            print(f"   Successfully Saved -> {output_name} ({len(final_df)} rows)")
        else:
            print(f"   ℹ️ {prefix} contained 0 coding exon records.")
            
    except Exception as e:
        print(f"❌ Error compiling data for {prefix}: {str(e)}")

print("\n🎉 Batch processing automation execution successfully finished!")

Found 108 files to process.
⚠️ Warning: Missing matching annotation file for ARRB1_coding. Skipping...
Processing: ARRB1...
   Successfully Saved -> arrb1_exon_coding_variants.csv (49 rows)
⚠️ Warning: Missing matching annotation file for ARRB2_coding. Skipping...
Processing: ARRB2...
   Successfully Saved -> arrb2_exon_coding_variants.csv (52 rows)
⚠️ Warning: Missing matching annotation file for CCKAR_coding. Skipping...
Processing: CCKAR...
   Successfully Saved -> cckar_exon_coding_variants.csv (49 rows)
⚠️ Warning: Missing matching annotation file for CCKBR_coding. Skipping...
Processing: CCKBR...
   Successfully Saved -> cckbr_exon_coding_variants.csv (69 rows)
⚠️ Warning: Missing matching annotation file for CCR2_coding. Skipping...
Processing: CCR2...
   Successfully Saved -> ccr2_exon_coding_variants.csv (50 rows)
⚠️ Warning: Missing matching annotation file for CCR5_coding. Skipping...
Processing: CCR5...
   Successfully Saved -> ccr5_exon_coding_variants.csv (42 rows)
⚠️ War

C:\Users\lenovo\AppData\Local\Temp\ipykernel_16236\481975432.py:50: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ann = pd.read_csv(ann_file, sep='\t', skiprows=36)


   Successfully Saved -> lpar1_exon_coding_variants.csv (54 rows)
⚠️ Warning: Missing matching annotation file for LPAR2_coding. Skipping...
Processing: LPAR2...
   Successfully Saved -> lpar2_exon_coding_variants.csv (36 rows)
⚠️ Warning: Missing matching annotation file for LPAR3_coding. Skipping...
Processing: LPAR3...
   Successfully Saved -> lpar3_exon_coding_variants.csv (46 rows)
⚠️ Warning: Missing matching annotation file for LPAR5_coding. Skipping...
Processing: LPAR5...
   Successfully Saved -> lpar5_exon_coding_variants.csv (57 rows)
⚠️ Warning: Missing matching annotation file for LPAR6_coding. Skipping...
Processing: LPAR6...
   Successfully Saved -> lpar6_exon_coding_variants.csv (52 rows)
⚠️ Warning: Missing matching annotation file for MRAP2_coding. Skipping...
Processing: MRAP2...
   Successfully Saved -> mrap2_exon_coding_variants.csv (21 rows)
⚠️ Warning: Missing matching annotation file for MRAP_coding. Skipping...
Processing: MRAP...
   Successfully Saved -> mrap_